In [24]:
import tarfile
import os

def create_model_package(model_files, inference_script, output_path):
    with tarfile.open(output_path, "w:gz") as tar:
        for file in model_files:
            print(os.path.basename(file))
            tar.add(file, arcname=os.path.join("code", os.path.basename(file)))
        # tar.add(inference_script)# arcname="code/inference.py")

# Usage
model_files = ["../blockhouse_ml","./inference.py", "requirements.txt"]
inference_script = ""
output_path = "model_new.tar.gz"
create_model_package(model_files, inference_script, output_path)


blockhouse_ml
inference.py
requirements.txt


In [10]:
# !tar -xvf model.tar.gz

In [25]:
import boto3

s3_client = boto3.client('s3')

version = '1'
bucket_name = 'sagemaker-tradingmodel-us-east-1'
s3_key = f'production/version_{version}/model_new.tar.gz'

s3_client.upload_file(output_path, bucket_name, s3_key)

In [ ]:
# ccreate session if local
local = False
import boto3
from sagemaker.local import LocalSession
if local:
    sagemaker_session = LocalSession()
    boto_session = sagemaker_session.boto_session
else:
    boto_session = boto3.Session()

In [ ]:

def create_sagemaker_model(model_name, role_arn, image_uri, model_data_url):
    sagemaker_client = boto_session.client('sagemaker')
    
    response = sagemaker_client.create_model(
        ModelName=model_name,
        PrimaryContainer={
            'Image': image_uri,
            'ModelDataUrl': model_data_url,
            'Environment': {
                'SAGEMAKER_PROGRAM': 'inference.py',
                'SAGEMAKER_CONTAINER_LOG_LEVEL': '10',
                'PYTHONUNBUFFERED':'1',
                # refernce :https://github.com/aws/sagemaker-python-sdk/issues/2574#issuecomment-1854585409
                'SAGEMAKER_TS_RESPONSE_TIMEOUT': '600', # 10 minutes
                'SAGEMAKER_MODEL_SERVER_TIMEOUT': '600', # 10 minutes
                # 'SAGEMAKER_SUBMIT_DIRECTORY': '/opt/ml'
                'S3_BUCKET': bucket_name,
                'PROD_VERSION': version,
            }
            
        },
        ExecutionRoleArn=role_arn,
        # EnableNetworkIsolation=False,
        # Mode="SingleModel",
    )
    
    return response

# Usage
model_name = 'model-real-time-inference3'
role_arn = "arn:aws:iam::433046797920:role/service-role/AmazonSageMaker-ExecutionRole-20240227T184832" #'arn:aws:iam::your-account-id:role/your-sagemaker-role'
image_uri = '433046797920.dkr.ecr.us-east-1.amazonaws.com/blockhouse-ml:deploy-test3'
model_data_url = f's3://{bucket_name}/{s3_key}'

response = create_sagemaker_model(model_name, role_arn, image_uri, model_data_url)
print(f"Model created: {response['ModelArn']}")

In [ ]:
def create_sagemaker_endpoint_config(model_name,  endpoint_config_name, endpoint_name, role_arn):
    sagemaker_client = boto_session.client('sagemaker')
    
    variant_name = 'variant-1'

    response = sagemaker_client.create_endpoint_config(
        EndpointConfigName=endpoint_config_name,
        ExecutionRoleArn = role_arn,
        ProductionVariants=[
            {
                'InstanceType': 'ml.g4dn.xlarge',
                'InitialInstanceCount': 1,
                "ManagedInstanceScaling": { 
                    "MaxInstanceCount": 1,
                    "MinInstanceCount": 1,
                },
                "VariantName": variant_name,
            }
        ]
    )
    
    sagemaker_client.create_endpoint(
        EndpointName=endpoint_name,
        EndpointConfigName=endpoint_config_name)
    
    response = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
    
    sagemaker_client.create_inference_component(
        InferenceComponentName=model_name+"-inference-component",
        EndpointName=endpoint_name,
        VariantName=variant_name,
        Specification={
            "ModelName": model_name, 
            "ComputeResourceRequirements": { 
                "NumberOfAcceleratorDevicesRequired": 1,
                "NumberOfCpuCoresRequired": 1, 
                "MinMemoryRequiredInMb": 128,
            }
        },
        RuntimeConfig={"CopyCount": 1},
    )
    # response["EndpointStatus"]
    return response

# Usage
model_name = 'model-real-time-inference3'
endpoint_config_name = 'endpoint-config-real-time-inference3'
endpoint_name = 'endpoint-real-time-inference3'

response = create_sagemaker_endpoint_config(model_name, endpoint_config_name, endpoint_name, role_arn)
print(f"Endpoint created: {response['EndpointArn']}")

In [ ]:
sagemaker_client = boto_session.client('sagemaker')
response = "Creating"
# while response == "Creating":
response = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
# if response["EndpointStatus"] == 'Creating': print(response["EndpointStatus"])
response = response["EndpointStatus"]
print(response)

In [16]:
## Run inference from endpoint (AWS) real time inference, this code will be used by front end app
import json
import boto3
# endpoint_name = "Model-1728065209105-deploy-test3-Endpoint-20241005-000648"
import time
# Prepare your JSON payload
payload = {
        "ticker": "JPM",
        "action": "buy",
        "inventory": 5000,
        "timeframe": 390
    }

# Endpoint name : 
start_time = time.time()

request_body = json.dumps(payload)
# Create a low-level client representing Amazon SageMaker Runtime
sagemaker_runtime = boto3.client(
    "sagemaker-runtime", region_name='us-east-1')
# Make the prediction

# Gets inference from the model hosted at the specified endpoint:
response = sagemaker_runtime.invoke_endpoint(
    EndpointName="endpoint-real-time-inference2424-HW-R", 
    Body=request_body, #bytes(request_body, 'utf-8')
    ContentType='application/json',
    InferenceComponentName="model-real-time-inference2424-HW-R-inference-component"
    )
end_time = time.time()

# Decodes and prints the response body:
print(response['Body'].read().decode('utf-8'))
print(f"Time taken for inference: {end_time - start_time} seconds")

[{"step": 0, "timestamp": "2024-10-14 15:28:00", "order_type": "Limit", "volume": 587, "limit_price": 221.24675575410012}, {"step": 1, "timestamp": "2024-10-15 09:30:00", "order_type": "Market", "volume": 1045, "limit_price": 221.51689146993814}, {"step": 2, "timestamp": "2024-10-15 10:20:00", "order_type": "Limit", "volume": 806, "limit_price": 221.7377465811255}, {"step": 3, "timestamp": "2024-10-15 11:10:00", "order_type": "Limit", "volume": 614, "limit_price": 221.17386469760825}, {"step": 4, "timestamp": "2024-10-15 12:00:00", "order_type": "Market", "volume": 606, "limit_price": 222.00062853425098}, {"step": 5, "timestamp": "2024-10-15 12:30:00", "order_type": "Limit", "volume": 543, "limit_price": 222.25615445653864}, {"step": 6, "timestamp": "2024-10-15 13:08:00", "order_type": "Market", "volume": 493, "limit_price": 222.32311991045952}, {"step": 7, "timestamp": "2024-10-15 13:58:00", "order_type": "Limit", "volume": 442, "limit_price": 221.8169125017681}]
Time taken for infere

In [3]:
# Inference testing locally....
import inference
import json
import os


version = '1'
bucket_name = 'sagemaker-tradingmodel-us-east-1'
os.environ['S3_BUCKET'] = bucket_name
os.environ['PROD_VERSION'] = version

model_dir = 'production_test'
os.makedirs(model_dir, exist_ok=True)

# Load the models from S3 and local storage
model = inference.model_fn(model_dir)

# Simulate a request body
request_body = json.dumps({
    "ticker": "JPM",
    "action": "sell",
    "inventory": 500,
    "timeframe": 300
})

import time
start_t = time.perf_counter()
# Run the inference process
input_data = inference.input_fn(request_body, 'application/json')
prediction = inference.predict_fn(input_data, model)
output = inference.output_fn(prediction, 'application/json')
end_t = time.perf_counter()
print(f"Time taken: {end_t - start_t}")

print("Prediction Result:", output)

ModuleNotFoundError: No module named 'inference'

In [ ]:
from sagemaker import image_uris
image_uris.retrieve(framework='pytorch',region='us-east-1',version='2.2.0',py_version='py310',image_scope='inference', instance_type='ml.g4dn.xlarge')



In [17]:
!python deployment.py --model_name "serverless-endpoint-trial-model-1" \
                     --account_id "433046797920" \
                     --version 1 \
                     --sagemaker_role "arn:aws:iam::433046797920:role/service-role/AmazonSageMakerServiceCatalogProductsUseRole" \
                     --model_files ../blockhouse_ml ./inference.py ./requirements.txt \
                     --endpoint_name "serverless-endpoint-trial-1"

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Starting deployment process...
Using image URI: 433046797920.dkr.ecr.us-east-1.amazonaws.com/blockhouse-ml:deploy-test3
Packaging model files: ['../blockhouse_ml', './inference.py', './requirements.txt']
Model package model_serverless.tar.gz created successfully.
Uploading model package to S3 bucket: sagemaker-tradingmodel-us-east-1, key: production/version_1/model_serverless.tar.gz
Model package uploaded to S3: s3://sagemaker-tradingmodel-us-east-1/production/version_1/model_serverless.tar.gz
Model data URL: s3://sagemaker-tradingmodel-us-east-1/production/version_1/model_serverless.tar.gz
Creating serverless inference configuration...
Serverless inference configuration created.
Creating SageMaker Model with model name: serverless-endpoint-trial-model-1
SageMaker Model created succe

In [16]:
import boto3
from botocore.exceptions import ClientError

sagemaker_client = boto3.client('sagemaker')

try:
    # Describe the endpoint to check the status after deployment
    endpoint_info = sagemaker_client.describe_endpoint(EndpointName="serverless-endpoint-trial-model-1")
    status = endpoint_info['EndpointStatus']
    print(f"Endpoint status: {status}")

    if status != 'InService':
        print(f"Endpoint failed with status: {status}. Check CloudWatch logs for more details.")
    else:
        print(f"Endpoint deployed successfully: serverless-endpoint-trial-model-1")

except ClientError as e:
    # Handle resource not found exception
    error_code = e.response['Error']['Code']
    if error_code == 'ValidationException':
        print("Endpoint serverless-endpoint-trial-model-1 not found. Deployment might have failed.")
    else:
        print(f"Unexpected error: {e}")


Endpoint serverless-endpoint-trial-model-1 not found. Deployment might have failed.


In [1]:
from sagemaker.serverless import ServerlessInferenceConfig
from sagemaker.model import Model

serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=4096,  # Adjust based on your model's memory requirements
    max_concurrency=5        # Number of concurrent invocations
)

model = Model(
    model_data='s3://sagemaker-tradingmodel-us-east-1/production/version_1/model.tar.gz',
    role='arn:aws:iam::433046797920:role/service-role/AmazonSageMakerServiceCatalogProductsUseRole',
    image_uri='433046797920.dkr.ecr.us-east-1.amazonaws.com/blockhouse-ml:deploy-test3',  # Optional if using a custom image
    entry_point='inference.py',
    source_dir='.',  # Ensure this contains blockhouse_ml and other files
)

predictor = model.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name="your-serverless-endpoint"
)

print(f"Serverless endpoint created successfully: {predictor.endpoint_name}")


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
----------------------*

Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-endpoint


UnexpectedStatusException: Error hosting endpoint your-serverless-endpoint: Failed. Reason: Image size 14642068847 is greater than supported size 10737418240. Try changing the instance type or reference the troubleshooting page https://docs.aws.amazon.com/sagemaker/latest/dg/async-inference-troubleshooting.html

In [27]:
from sagemaker.serverless import ServerlessInferenceConfig
from sagemaker.pytorch import PyTorchModel

# Define serverless configuration
serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=6144,  # Adjust based on your model's memory requirements
    max_concurrency=5        # Number of concurrent invocations
)

# Define the model with SageMaker's pre-built PyTorch container
model = PyTorchModel(
    model_data='s3://sagemaker-tradingmodel-us-east-1/production/version_1/model_new.tar.gz',
    role='arn:aws:iam::433046797920:role/service-role/AmazonSageMakerServiceCatalogProductsUseRole',
    entry_point='inference.py',  # Ensure your inference script is set correctly
    source_dir='.',  # Ensure this contains blockhouse_ml, requirements.txt, and other files
    framework_version='2.1.0',  # Version of PyTorch you want to use
    py_version='py310'  # Python version
)

# Deploy the model as a serverless endpoint
predictor = model.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name="my-new-serverless-endpoint-9"
)

print(f"Serverless endpoint created successfully: {predictor.endpoint_name}")


--------!Serverless endpoint created successfully: my-new-serverless-endpoint-9


In [28]:
import boto3

# Define SageMaker runtime client
sagemaker_runtime = boto3.client('sagemaker-runtime', region_name='us-east-1')

# Define the input for the model
payload = json.dumps({
    "ticker": "JPM",
    "action": "buy",
    "inventory": 1000,
    "timeframe": 300
})

# Invoke the endpoint
response = sagemaker_runtime.invoke_endpoint(
    EndpointName="my-new-serverless-endpoint-9",
    ContentType="application/json",
    Body=payload
)

# Print the result
result = json.loads(response['Body'].read().decode())
print(f"Inference result: {result}")


ReadTimeoutError: Read timeout on endpoint URL: "https://runtime.sagemaker.us-east-1.amazonaws.com/endpoints/my-new-serverless-endpoint-9/invocations"